<a href="https://colab.research.google.com/github/2026-2-Generative-AI-Project/Segmentation-SAM/blob/feature%2Fnukki/adcanvas_nukki_Fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. 설치

In [ ]:
!pip install -q timm kornia einops pillow-heif
!SAM2_BUILD_CUDA=0 pip install -q git+https://github.com/facebookresearch/sam2.git
!pip install -q ultralytics jupyter_bbox_widget pillow-heif

# 1. 경로와 설정

In [ ]:
import os, glob, json, time, random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pillow_heif import register_heif_opener
from scipy import ndimage
from google.colab import drive, userdata

register_heif_opener()   # 아이폰 HEIC
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/AdCanvas_nukki'
IN_DIR = f'{BASE}/images'               # 입력 사진
GT_DIR = f'{BASE}/gt'                   # 정답 마스크 (사진과 같은 이름의 .png)
SEG_DIR = f'{BASE}/seg_pipeline'        # 이 노트북의 결과
RMBG_DIR = f'{SEG_DIR}/rmbg_handoff'    # RMBG 1차 마스크 + 박스 → 파인튜닝 담당자에게 전달
W_DIR = f'{BASE}/weights'               # 파인튜닝된 SAM 가중치를 놓는 곳
for d in (IN_DIR, GT_DIR, SEG_DIR, RMBG_DIR, W_DIR):
    os.makedirs(d, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'

CFG = dict(
    max_side=2048,        # 처리 해상도 (긴 변)
    rmbg_th=0.5,          # RMBG 확률 마스크 → 이진화 기준
    min_area=0.003,       # 사진 넓이 대비 이보다 작은 덩어리는 부스러기로 보고 무시
    box_pad=0.05,         # 박스를 사방으로 늘리는 비율 (RMBG가 덜 잡은 가장자리 보완)
    max_boxes=15,         # 박스 최대 개수 (큰 덩어리부터)
    sam_model='facebook/sam2.1-hiera-small',
    sam_ft_weights=f'{W_DIR}/sam2_ft.pt',   # 파인튜닝 결과. 없으면 B3 건너뜀
    eval_side=1024,       # 평가 해상도 (긴 변)
    bf_tol=0.0075,        # Boundary F1 허용 거리 (사진 대각선 길이 대비)
    test_ratio=0.3,       # 시험용 비율
    seed=42,
)
CATEGORIES = ['normal', 'white', 'multi', 'hand_tray']   # 일반 / 흰 음식 / 여러 음식 / 손·쟁반
EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif')


def list_images():
    return sorted(p for p in glob.glob(f'{IN_DIR}/*') if p.lower().endswith(EXTS))


def stem(p):
    return os.path.splitext(os.path.basename(p))[0]


def load_image(p, max_side=CFG['max_side']):
    img = ImageOps.exif_transpose(Image.open(p)).convert('RGB')
    if max_side:
        img.thumbnail((max_side, max_side), Image.LANCZOS)
    return img


print('device:', device, '| 사진', len(list_images()), '장 | 정답', len(glob.glob(f'{GT_DIR}/*.png')), '장')

# 2. 사진 분류와 학습/테스트 나누기
* 처음 실행하면 meta.csv가 만들어지고, 드라이브에서 열어 category 칸을 normal, white, multi, hand_tray 중 하나로 고친 뒤 이 셀을 다시 실행
* split.csv는 한번 만들어지면 절대 바뀌지 않음. 파인튜닝 전후 비교를 위한 값
* 나중에 정답이 추가된 사진은 자동으로 학습용으로 들어감

In [ ]:
META = f'{BASE}/meta.csv'
SPLIT = f'{BASE}/split.csv'

names = [stem(p) for p in list_images()]
meta = pd.read_csv(META) if os.path.exists(META) else pd.DataFrame(columns=['image', 'category'])
new = [n for n in names if n not in set(meta.image)]
if new:
    meta = pd.concat([meta, pd.DataFrame({'image': new, 'category': 'normal'})], ignore_index=True)
    meta.to_csv(META, index=False)
    print(f'meta.csv에 {len(new)}장 추가 (category=normal). 드라이브에서 열어 분류를 고친 뒤 이 셀을 다시 실행하세요.')
bad = meta[~meta.category.isin(CATEGORIES)]
assert bad.empty, f'category 오타: {bad.to_dict("records")}'

# 정답이 있는 사진만 나눔. 처음 한 번은 분류별로 같은 비율이 되게 나눔
has_gt = {stem(p) for p in glob.glob(f'{GT_DIR}/*.png')}
split = pd.read_csv(SPLIT) if os.path.exists(SPLIT) else pd.DataFrame(columns=['image', 'split'])
todo = meta[meta.image.isin(has_gt) & ~meta.image.isin(set(split.image))]
if len(todo):
    rows = []
    if split.empty:
        rng = random.Random(CFG['seed'])
        for cat, g in todo.groupby('category'):
            imgs = sorted(g.image)
            rng.shuffle(imgs)
            n_test = round(len(imgs) * CFG['test_ratio']) if len(imgs) >= 2 else 0
            rows += [{'image': n, 'split': 'test' if i < n_test else 'train'} for i, n in enumerate(imgs)]
    else:
        rows = [{'image': n, 'split': 'train'} for n in todo.image]
    split = pd.concat([split, pd.DataFrame(rows)], ignore_index=True)
    split.to_csv(SPLIT, index=False)
print(split.merge(meta, on='image').groupby(['split', 'category']).size().unstack(fill_value=0))

# 라벨링

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam_base = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)


def tint(img_np, m, color=(255, 0, 0)):
    out = img_np.copy()
    out[m] = (out[m] * 0.5 + np.array(color) * 0.5).astype(np.uint8)
    return out

# 3. 모델 불러오기
* 허깅페이스 토큰은 코랩 왼쪽에 HF_TOKEN이라는 이름으로 저장해 두면 자동으로 쓰임

In [ ]:
from huggingface_hub import login
from torchvision import transforms
from transformers import AutoModelForImageSegmentation
from sam2.sam2_image_predictor import SAM2ImagePredictor

try:
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

# RMBG-2.0
rmbg = AutoModelForImageSegmentation.from_pretrained('briaai/RMBG-2.0', trust_remote_code=True).to(device).eval()
rmbg_tf = transforms.Compose([
    transforms.Resize((1024, 1024)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


@torch.no_grad()
def rmbg_mask(img):
    """RMBG-2.0 확률 마스크 (H, W) 0~1"""
    x = rmbg_tf(img).unsqueeze(0).to(device, next(rmbg.parameters()).dtype)
    pred = rmbg(x)[-1].sigmoid()[0, 0].float().cpu()
    pil = transforms.functional.to_pil_image(pred).resize(img.size, Image.BILINEAR)
    return np.asarray(pil, dtype=np.float32) / 255


# SAM 2.1 원본 (B2)
sam_base = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)

# SAM 2.1 파인튜닝 (B3): 가중치 파일이 있을 때만
sam_ft = None
if os.path.exists(CFG['sam_ft_weights']):
    sam_ft = SAM2ImagePredictor.from_pretrained(CFG['sam_model'], device=device)
    sd = torch.load(CFG['sam_ft_weights'], map_location=device)
    res = sam_ft.model.load_state_dict(sd, strict=False)
    print('파인튜닝 가중치 불러옴 | 없는 키', len(res.missing_keys), '| 남는 키', len(res.unexpected_keys))
else:
    print('파인튜닝 가중치 없음 → B3 조건은 건너뜀:', CFG['sam_ft_weights'])

# 4. 파이프라인 함수


In [ ]:
def boxes_from_mask(m_bin, cfg=CFG):
    """RMBG 이진 마스크 → 덩어리마다 박스 [x0, y0, x1, y1] (큰 덩어리부터)"""
    H, W = m_bin.shape
    lab, n = ndimage.label(m_bin)
    if n == 0:
        return []
    sizes = ndimage.sum(m_bin, lab, range(1, n + 1))
    cand = []
    for i, sl in enumerate(ndimage.find_objects(lab)):
        if sizes[i] < cfg['min_area'] * H * W:
            continue
        y0, y1, x0, x1 = sl[0].start, sl[0].stop, sl[1].start, sl[1].stop
        ph, pw = (y1 - y0) * cfg['box_pad'], (x1 - x0) * cfg['box_pad']
        cand.append((sizes[i], [max(0, x0 - pw), max(0, y0 - ph), min(W, x1 + pw), min(H, y1 + ph)]))
    cand.sort(key=lambda c: -c[0])
    return [[float(v) for v in b] for _, b in cand[:cfg['max_boxes']]]


@torch.no_grad()
def sam_refine(predictor, img_np, boxes):
    """박스마다 SAM 실행 → 박스 밖은 잘라내고 → 합치기"""
    H, W = img_np.shape[:2]
    out = np.zeros((H, W), dtype=bool)
    if not boxes:
        return out
    predictor.set_image(img_np)
    masks, _, _ = predictor.predict(box=np.array(boxes, dtype=np.float32), multimask_output=False)
    masks = np.asarray(masks).reshape(-1, H, W) > 0.5
    for (x0, y0, x1, y1), m in zip(boxes, masks):
        clip = np.zeros_like(m)
        clip[int(y0):int(np.ceil(y1)), int(x0):int(np.ceil(x1))] = True
        out |= m & clip
    return out


def segment(img, mode='rmbg+sam'):
    """통합 함수 (나중에 FastAPI의 pipeline/segment.py로 옮길 부분)
    mode: 'rmbg' | 'rmbg+sam' | 'rmbg+sam_ft'
    반환: (마스크 'L' 이미지, 정보 dict)"""
    soft = rmbg_mask(img)
    boxes = boxes_from_mask(soft > CFG['rmbg_th'])
    if mode == 'rmbg' or not boxes:          # RMBG가 음식을 못 찾으면 RMBG 결과 그대로
        m = soft
    else:
        pred = sam_ft if mode == 'rmbg+sam_ft' else sam_base
        assert pred is not None, '파인튜닝 가중치가 없습니다'
        m = sam_refine(pred, np.array(img), boxes).astype(np.float32)
    mask = Image.fromarray((m * 255).astype(np.uint8))
    return mask, {'boxes': boxes, 'n_boxes': len(boxes), 'no_food': not boxes}

# 5. 사진 한장으로 확인해보기
- 디버깅용, 최종 코드에서는 삭제

In [ ]:
def tint(img_np, m, color=(255, 0, 0)):
    out = img_np.copy()
    out[m] = (out[m] * 0.5 + np.array(color) * 0.5).astype(np.uint8)
    return out


def show_one(name):
    p = [q for q in list_images() if stem(q) == name][0]
    img = load_image(p)
    base = np.array(img)
    soft = rmbg_mask(img)
    boxes = boxes_from_mask(soft > CFG['rmbg_th'])
    sam_m = sam_refine(sam_base, base, boxes)

    fig, ax = plt.subplots(1, 4, figsize=(20, 5))
    ax[0].imshow(img); ax[0].set_title(f'boxes from RMBG ({len(boxes)})')
    for x0, y0, x1, y1 in boxes:
        ax[0].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color='lime', lw=2))
    ax[1].imshow(soft, cmap='gray'); ax[1].set_title('B1 RMBG-2.0')
    ax[2].imshow(sam_m, cmap='gray'); ax[2].set_title('B2 RMBG -> SAM')
    ax[3].imshow(tint(base, sam_m)); ax[3].set_title('B2 overlay')
    for a in ax:
        a.axis('off')
    plt.tight_layout(); plt.show()


show_one(names[0])   # 다른 사진: show_one('IMG_1190')

# 6. 전체 사진 실행

In [ ]:
CONDS = {'B1_rmbg': None, 'B2_rmbg_sam': sam_base}
if sam_ft is not None:
    CONDS['B3_rmbg_sam_ft'] = sam_ft
for c in CONDS:
    os.makedirs(f'{SEG_DIR}/{c}', exist_ok=True)

handoff, times = {}, []
for i, p in enumerate(list_images()):
    name, img = stem(p), load_image(p)
    base = np.array(img)

    t0 = time.perf_counter()
    soft = rmbg_mask(img)
    boxes = boxes_from_mask(soft > CFG['rmbg_th'])
    t_rmbg = time.perf_counter() - t0

    # 파인튜닝 담당자용: RMBG 마스크 + 박스 (픽셀 좌표와 0~1 비율 좌표 둘 다)
    Image.fromarray((soft * 255).astype(np.uint8)).save(f'{RMBG_DIR}/{name}.png')
    W, H = img.size
    handoff[name] = {'size': [W, H], 'boxes': boxes,
                     'boxes_norm': [[b[0] / W, b[1] / H, b[2] / W, b[3] / H] for b in boxes]}

    for c, pred in CONDS.items():
        t0 = time.perf_counter()
        m = soft if (pred is None or not boxes) else sam_refine(pred, base, boxes).astype(np.float32)
        dt = time.perf_counter() - t0
        Image.fromarray((m * 255).astype(np.uint8)).save(f'{SEG_DIR}/{c}/{name}.png')
        times.append({'image': name, 'cond': c, 'n_boxes': len(boxes),
                      'sec': round(t_rmbg + (dt if pred is not None else 0), 3)})
    print(f'[{i + 1}] {name}: 박스 {len(boxes)}개' + ('  ← RMBG가 음식을 못 찾음' if not boxes else ''))

json.dump({'cfg': CFG, 'images': handoff}, open(f'{RMBG_DIR}/boxes.json', 'w'), ensure_ascii=False, indent=1)
tdf = pd.DataFrame(times)
tdf.to_csv(f'{SEG_DIR}/seg_time.csv', index=False)
print(tdf.groupby('cond')['sec'].agg(['mean', 'max']).round(3))

# 7. 평가

In [ ]:
def to_eval(mask_img, size):
    return np.asarray(mask_img.convert('L').resize(size, Image.BILINEAR), dtype=np.float32) / 255


def boundary_f1(p, g, tol):
    """경계선끼리 tol 픽셀 이내로 맞는 비율 (경계 품질)"""
    pb = p ^ ndimage.binary_erosion(p)
    gb = g ^ ndimage.binary_erosion(g)
    if not pb.any() and not gb.any():
        return 1.0
    if not pb.any() or not gb.any():
        return 0.0
    prec = (ndimage.distance_transform_edt(~gb)[pb] <= tol).mean()
    rec = (ndimage.distance_transform_edt(~pb)[gb] <= tol).mean()
    return float(2 * prec * rec / (prec + rec + 1e-8))


def metrics(p_soft, g):
    pb = p_soft > 0.5
    union = (pb | g).sum()
    tol = CFG['bf_tol'] * np.hypot(*g.shape)
    return {'IoU': float((pb & g).sum() / union) if union else 1.0,
            'BF': boundary_f1(pb, g, tol),
            'MAE': float(np.abs(p_soft - g).mean())}


rows = []
info = split.merge(meta, on='image')
for _, r in info.iterrows():
    gt_img = Image.open(f'{GT_DIR}/{r.image}.png')
    s = CFG['eval_side'] / max(gt_img.size)
    size = (round(gt_img.width * s), round(gt_img.height * s))
    g = to_eval(gt_img, size) > 0.5
    for c in CONDS:
        mp = f'{SEG_DIR}/{c}/{r.image}.png'
        if os.path.exists(mp):
            rows.append({'image': r.image, 'split': r.split, 'category': r.category, 'cond': c,
                         **metrics(to_eval(Image.open(mp), size), g)})

mdf = pd.DataFrame(rows)
mdf.to_csv(f'{SEG_DIR}/seg_metrics.csv', index=False)
test = mdf[mdf.split == 'test']
print('=== test 전체 ===')
print(test.groupby('cond')[['IoU', 'BF', 'MAE']].mean().round(4))
print('\n=== test 분류별 IoU ===')
print(test.pivot_table(index='category', columns='cond', values='IoU', aggfunc='mean').round(3))
print('\n=== test 분류별 BF ===')
print(test.pivot_table(index='category', columns='cond', values='BF', aggfunc='mean').round(3))

# 8. 전후 샘플 그림(보고서용)

In [ ]:
def plot_samples(img_names, save_name):
    cols = ['image', 'GT'] + list(CONDS)
    fig, axes = plt.subplots(len(img_names), len(cols),
                             figsize=(3.2 * len(cols), 3.2 * len(img_names)), squeeze=False)
    for r, name in enumerate(img_names):
        p = [q for q in list_images() if stem(q) == name][0]
        img = load_image(p, CFG['eval_side'])
        base = np.array(img)
        g = to_eval(Image.open(f'{GT_DIR}/{name}.png'), img.size) > 0.5
        for c, col in enumerate(cols):
            ax = axes[r][c]
            if col == 'image':
                ax.imshow(img); ax.set_title(name, fontsize=9)
            elif col == 'GT':
                ax.imshow(tint(base, g, (0, 200, 0))); ax.set_title('GT', fontsize=9)
            else:
                m = to_eval(Image.open(f'{SEG_DIR}/{col}/{name}.png'), img.size) > 0.5
                iou = mdf[(mdf.image == name) & (mdf.cond == col)].IoU
                ax.imshow(tint(base, m))
                ax.set_title(f'{col}\nIoU {iou.iloc[0]:.3f}' if len(iou) else col, fontsize=9)
            ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'{SEG_DIR}/{save_name}', dpi=120, bbox_inches='tight')
    plt.show()


test_names = sorted(test.image.unique())
plot_samples(test_names[:12], 'samples_test.png')            # 시험용 사진 샘플

worst = test[test.cond == 'B2_rmbg_sam'].sort_values('IoU').image.head(6).tolist()
plot_samples(worst, 'samples_worst.png')                      # 가장 못한 사진 (한계 분석용)

# 9. 최종 출력(VLM으로 넘길 PNG)

In [ ]:
import io
from PIL import ImageFilter

# 파인튜닝 가중치가 있으면 B3, 없으면 B2 결과를 최종으로 사용
FINAL_COND = 'B3_rmbg_sam_ft' if 'B3_rmbg_sam_ft' in CONDS else 'B2_rmbg_sam'
FINAL_DIR = f'{BASE}/final'
os.makedirs(FINAL_DIR, exist_ok=True)


def make_cutout(img, mask, feather=1):
    """배경이 투명한 음식 누끼 (RGBA). feather: 경계를 살짝 부드럽게"""
    a = mask.convert('L')
    if feather:
        a = a.filter(ImageFilter.GaussianBlur(feather))
    out = img.convert('RGBA')
    out.putalpha(a)
    return out


def to_png_bytes(img):
    """FastAPI로 주고받을 때 쓰는 PNG 바이트"""
    buf = io.BytesIO()
    img.save(buf, 'PNG')
    return buf.getvalue()


manifest = {}
for p in list_images():
    name, img = stem(p), load_image(p)
    mask = Image.open(f'{SEG_DIR}/{FINAL_COND}/{name}.png').convert('L')
    cut = make_cutout(img, mask)
    bbox = mask.point(lambda v: 255 if v > 127 else 0).getbbox()   # 음식이 있는 영역

    img.save(f'{FINAL_DIR}/{name}_image.png')      # 원본 (HEIC도 PNG로 변환, 회전 반영)
    mask.save(f'{FINAL_DIR}/{name}_mask.png')      # 흑백 마스크 (흰색 = 음식)
    cut.save(f'{FINAL_DIR}/{name}_cut.png')        # 투명 배경 누끼
    if bbox:
        cut.crop(bbox).save(f'{FINAL_DIR}/{name}_cut_crop.png')   # 음식 부분만 잘라낸 누끼

    manifest[name] = {
        'image': f'{name}_image.png', 'mask': f'{name}_mask.png',
        'cut': f'{name}_cut.png', 'cut_crop': f'{name}_cut_crop.png' if bbox else None,
        'size': list(img.size), 'food_bbox': list(bbox) if bbox else None,
        'seg_cond': FINAL_COND, 'no_food': bbox is None,
    }

json.dump(manifest, open(f'{FINAL_DIR}/manifest.json', 'w'), ensure_ascii=False, indent=1)
print(f'{len(manifest)}장 저장 → {FINAL_DIR}  (사용한 조건: {FINAL_COND})')
print('음식을 못 찾은 사진:', [n for n, v in manifest.items() if v['no_food']])

# 정답 마스크 만들기
